<div align="center">

# 🔍 **TelegramUserCheckBot**
### *Telegram Username Availability Checker*

---

Check Telegram username availability at scale — right from your browser.

**⚡ No install needed • 🤖 Telegram alerts • 🚀 Multi-threaded**

[![GitHub](https://img.shields.io/badge/GitHub-Repo-181717?logo=github)](https://github.com/Shineii86/TelegramUserCheckBot)
[![License](https://img.shields.io/badge/License-MIT-green.svg)](https://opensource.org/licenses/MIT)

</div>

---

## 🧠 How It Works

```
┌─────────────────┐     ┌──────────────┐     ┌───────────────┐
│  Generate / Load │────▶│  Check t.me  │────▶│  Parse Result │
│   Usernames      │     │   Pages      │     │  (HTML)       │
└─────────────────┘     └──────────────┘     └───────┬───────┘
                                                      │
                              ┌────────────────────────┼────────────────┐
                              ▼                        ▼                ▼
                        ┌──────────┐            ┌──────────┐      ┌──────────┐
                        │ ✅ Available│          │ ❌ Taken  │      │ ⚠️ Error  │
                        └──────────┘            └──────────┘      └──────────┘
```

**Detection method:** Scrapes `t.me/{username}` pages and checks for:
- 📸 Profile photo → **Taken**
- 👤 Display name → **Taken**
- 📊 Subscriber count → **Taken** (channel/group)
- 📝 Bio text → **Taken**
- No profile data → **Available** ✅

---

## 📦 Step 1 — Setup

In [ ]:
#@title 🚀 Install & Clone Repository
#@markdown *Run this cell first — installs dependencies and clones the repo.*

import os, sys, subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"❌ Error: {result.stderr.strip()}")
    return result

if not os.path.exists("TelegramUserCheckBot"):
    print("📥 Cloning repository...")
    run("git clone https://github.com/Shineii86/TelegramUserCheckBot.git")
else:
    print("📥 Pulling latest changes...")
    run("cd TelegramUserCheckBot && git pull")

print("📦 Installing dependencies...")
run("pip install -r TelegramUserCheckBot/requirements.txt -q")

sys.path.insert(0, "TelegramUserCheckBot")
os.chdir("TelegramUserCheckBot")

print()
print("═" * 50)
print("  ✅ Setup complete! Proceed to Step 2.")
print("═" * 50)

---

## ⚙️ Step 2 — Configuration

Fill in your details below. **Bot Token** and **Chat ID** are required for Telegram alerts.

| Setting | Description |
|---------|-------------|
| `BOT_TOKEN` | Get from [@BotFather](https://t.me/BotFather) → `/newbot` |
| `CHAT_ID` | Get from [@userinfobot](https://t.me/userinfobot) |
| `MODE` | `hits` = stop after N finds, `count` = stop after N checks, `continuous` = run forever |
| `USERNAME_LENGTH` | 5–32 chars (Telegram rule) |
| `DELAY` | Seconds between requests (higher = safer) |

In [ ]:
#@title 🔧 Configure Settings

#@markdown ### 🔑 Credentials
BOT_TOKEN = ""  #@param {type:"string"}
CHAT_ID = ""    #@param {type:"string"}

#@markdown ---
#@markdown ### 🎯 Check Mode
MODE = "hits"  #@param ["hits", "count", "continuous"]
#@markdown *`hits` = stop after finding N available names · `count` = check exactly N names · `continuous` = run forever*
STOP_AFTER = 10  #@param {type:"slider", min:1, max:100, step:1}
MAX_ATTEMPTS = 100  #@param {type:"slider", min:10, max:1000, step:10}

#@markdown ---
#@markdown ### 🎲 Username Generation
USERNAME_LENGTH = 5  #@param {type:"slider", min:5, max:32, step:1}

#@markdown ---
#@markdown ### 🚀 Performance
WORKERS = 10  #@param {type:"slider", min:1, max:50, step:1}
DELAY = 1.0   #@param {type:"slider", min:0.5, max:5.0, step:0.5}

#@markdown ---
#@markdown ### 🌐 Proxy (Optional)
USE_PROXIES = False  #@param {type:"boolean"}
PROXY_URL = ""       #@param {type:"string"}

#@markdown ---
#@markdown ### 📁 Wordlist (Optional)
USE_WORDLIST = False  #@param {type:"boolean"}
WORDLIST_URL = ""     #@param {type:"string"}

import os
os.environ["TELEGRAM_BOT_TOKEN"] = BOT_TOKEN
os.environ["TELEGRAM_CHAT_ID"] = CHAT_ID
os.environ["MODE"] = MODE
os.environ["STOP_AFTER_HITS"] = str(STOP_AFTER)
os.environ["MAX_ATTEMPTS"] = str(MAX_ATTEMPTS)
os.environ["USERNAME_LENGTH"] = str(USERNAME_LENGTH)
os.environ["MAX_WORKERS"] = str(WORKERS)
os.environ["DELAY"] = str(DELAY)
os.environ["USE_PROXIES"] = str(USE_PROXIES).lower()
os.environ["PROXY_URL"] = PROXY_URL
os.environ["USE_WORDLIST"] = str(USE_WORDLIST).lower()
os.environ["WORDLIST_URL"] = WORDLIST_URL

print("═" * 50)
print("  ⚙️  Configuration Summary")
print("═" * 50)
token_status = '✅ Set' if BOT_TOKEN else '❌ Missing'
print(f"  🔑 Token:      {token_status}")
chat_status = '✅ Set' if CHAT_ID else '❌ Missing'
print(f"  💬 Chat ID:    {chat_status}")
print(f"  🎯 Mode:       {MODE}")
if MODE == "hits":
    print(f"  🛑 Stop after: {STOP_AFTER} hits")
elif MODE == "count":
    print(f"  🔢 Max checks: {MAX_ATTEMPTS}")
print(f"  📏 Length:      {USERNAME_LENGTH} chars")
print(f"  🧵 Workers:    {WORKERS}")
print(f"  ⏱️  Delay:      {DELAY}s")
proxy_status = '✅ Enabled' if USE_PROXIES else '❌ Disabled'
print(f"  🌐 Proxies:    {proxy_status}")
wordlist_status = '✅ Enabled' if USE_WORDLIST else '❌ Disabled'
print(f"  📁 Wordlist:   {wordlist_status}")
print("═" * 50)

if not BOT_TOKEN or not CHAT_ID:
    print("\n⚠️  BOT_TOKEN and CHAT_ID are required for Telegram alerts.")
    print("   Without them, results will only show in this notebook.")
    print("   Get token: https://t.me/BotFather → /newbot")
    print("   Get ID:    https://t.me/userinfobot")

---

## 🔍 Step 3 — Run the Checker (CLI Mode)

Runs the multi-threaded checker directly in this notebook. Results saved to `available_usernames.txt`.

In [ ]:
#@title 🚀 Run Username Checker
#@markdown *Starts checking based on your Step 2 configuration.*

import os, sys, signal
sys.path.insert(0, ".")

from checker.config import Config
from checker.core import Checker

config = Config.from_env()
errors = config.validate()
if errors:
    for e in errors:
        print(f"❌ {e}")
else:
    print("🔍 Starting TelegramUserCheckBot...")
    print("═" * 50)
    print(f"  Mode: {config.mode} | Workers: {config.max_workers} | Delay: {config.delay}s")
    print("═" * 50)
    print()

    checker = Checker(config)
    signal.signal(signal.SIGINT, lambda *_: checker.stop())
    stats = checker.run()

    print()
    print("═" * 50)
    print("  📊 Results Summary")
    print("═" * 50)
    print(f"  ✅ Checked:    {stats.checked}")
    print(f"  🎯 Available:  {stats.hits}")
    print(f"  ❌ Taken:      {stats.taken}")
    print(f"  🚫 Invalid:    {stats.invalid}")
    print(f"  ⚠️  Rate Limit: {stats.rate_limited}")
    print(f"  💥 Errors:     {stats.errors}")
    print("═" * 50)

    if stats.available:
        print(f"\n🎯 Available usernames ({len(stats.available)}):")
        for u in stats.available:
            print(f"  → @{u}")
        print(f"\n💾 Saved to: available_usernames.txt")

---

## 🤖 Step 3B — Run Telegram Bot

Want the **interactive bot** instead? This starts a Telegram bot you can chat with directly.

**What you get:**
- `/check`, `/batch`, `/generate` commands
- Inline keyboard settings
- Quick check by just typing a username
- Instant hit notifications on Telegram

⚠️ **This cell will keep running** — that's normal! The bot stays alive while this cell runs.
**To stop the bot:** click the ⏹️ (Stop) button on the cell.

In [ ]:
#@title 🤖 Launch Telegram Bot
#@markdown ---
#@markdown ### 🔑 Paste your Bot Token
#@markdown Get one from [@BotFather](https://t.me/BotFather) → `/newbot`
BOT_TOKEN_HERE = ""  #@param {type:"string"}

#@markdown ---
#@markdown ### ▶️ Then run this cell and open your bot in Telegram!

import os, sys
from IPython.display import display, HTML
sys.path.insert(0, ".")

# Fix event loop for Jupyter/Colab
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nest_asyncio"])
    import nest_asyncio
    nest_asyncio.apply()

if not BOT_TOKEN_HERE:
    print("❌ Please enter your BOT_TOKEN above.")
    print("   Get one from: https://t.me/BotFather → /newbot")
else:
    os.environ["TELEGRAM_BOT_TOKEN"] = BOT_TOKEN_HERE

    # Show a nice info box
    display(HTML('''
    <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white;
                padding: 20px; border-radius: 12px; font-family: sans-serif; margin: 10px 0;">
        <h3 style="margin:0 0 10px 0;">🤖 Telegram Bot is Starting...</h3>
        <p style="margin:0; opacity:0.9; line-height: 1.8;">
            ✅ Open <b>Telegram</b> and find your bot<br>
            ✅ Send <code>/start</code> to begin<br>
            ✅ Try: <code>/check username</code> or just type a username<br>
            ✅ <b>Keep this cell running!</b> Stopping it kills the bot
        </p>
    </div>
    '''))

    print("
" + "═" * 50)
    print("  🤖 TelegramUserCheckBot is LIVE")
    print("═" * 50)
    print("  📱 Commands: /start /check /batch /generate /pattern /settings /stats /stop")
    print("  💡 Quick:    Just type any username to check it")
    print("  🛑 Stop:    Click ⏹️ on this cell")
    print("═" * 50)
    print()

    try:
        from bot.handlers import run_bot
        run_bot(BOT_TOKEN_HERE)
    except KeyboardInterrupt:
        print("
🛑 Bot stopped.")
    except Exception as e:
        print(f"
❌ Error: {e}")
        print("   Make sure your token is valid (get one from @BotFather).")


---

## 🧪 Step 4 — Quick Single Check

Just want to check one username? Use this:

In [ ]:
#@title 🔍 Check a Single Username

USERNAME_TO_CHECK = ""  #@param {type:"string"}
#@markdown *Enter a username without the @ symbol (e.g., `myusername`)*

import sys
sys.path.insert(0, ".")

from checker.telegram_client import TelegramUsernameClient, is_valid_username

if not USERNAME_TO_CHECK:
    print("⚠️ Please enter a username above.")
else:
    username = USERNAME_TO_CHECK.strip().lower().lstrip("@")

    if not is_valid_username(username):
        print(f"🚫 @{username} is not a valid Telegram username.")
        print("   Rules: 5-32 chars, a-z/0-9/_ only, starts with letter, no double underscores")
    else:
        print(f"🔍 Checking @{username}...")
        print()

        client = TelegramUsernameClient(user_agents=[
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
        ])
        status, _ = client.check(username, delay=0)

        if status == "available":
            print(f"  ✅ @{username} is AVAILABLE!")
            print(f"  🔗 https://t.me/{username}")
            print()
            print("  💡 Claim it now before someone else does!")
        elif status == "taken":
            print(f"  ❌ @{username} is already taken.")
        elif status == "invalid":
            print(f"  🚫 @{username} is not a valid username.")
        elif status == "rate_limited":
            print(f"  ⚠️ Rate limited. Try again in a few seconds.")
        else:
            print(f"  💥 Error checking @{username}. Try again.")

---

## 📋 Step 5 — Check Multiple Usernames

Enter usernames in the text box below (one per line), then click **Run Check**.

In [ ]:
#@title 📋 Check Multiple Usernames

import sys, ipywidgets as widgets
from IPython.display import display
sys.path.insert(0, ".")

print("📋 Enter usernames below (one per line), then click 'Run Check':")
print("═" * 50)

text_area = widgets.Textarea(
    value='telegram\nduvo\ngoogle\nmytestname123',
    placeholder='Enter usernames, one per line...',
    description='Usernames:',
    layout=widgets.Layout(width='100%', height='160px'),
    style={'description_width': '80px'}
)
display(text_area)

#@markdown ---
DELAY_BETWEEN = 1.5  #@param {type:"number"}
#@markdown *Seconds between each check. Higher = safer from rate limits.*

from checker.telegram_client import TelegramUsernameClient, is_valid_username

def run_batch_check(_):
    usernames = [u.strip().lower().lstrip("@") for u in text_area.value.strip().split("\n") if u.strip()]

    if not usernames:
        print("⚠️ Please enter usernames above (one per line).")
        return

    client = TelegramUsernameClient(user_agents=[
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
    ])

    available, taken, errors = [], [], []

    print(f"\n🔍 Checking {len(usernames)} usernames...")
    print("═" * 50)

    for i, username in enumerate(usernames, 1):
        if not is_valid_username(username):
            print(f"  [{i}/{len(usernames)}] 🚫 @{username} — invalid")
            errors.append(username)
            continue

        status, _ = client.check(username, delay=DELAY_BETWEEN)

        if status == "available":
            print(f"  [{i}/{len(usernames)}] ✅ @{username} — AVAILABLE")
            available.append(username)
        elif status == "taken":
            print(f"  [{i}/{len(usernames)}] ❌ @{username} — taken")
            taken.append(username)
        else:
            print(f"  [{i}/{len(usernames)}] ⚠️ @{username} — {status}")
            errors.append(username)

    print("═" * 50)
    print(f"\n📊 Results:")
    print(f"  ✅ Available: {len(available)}")
    print(f"  ❌ Taken: {len(taken)}")
    print(f"  ⚠️ Errors: {len(errors)}")

    if available:
        print(f"\n🎯 Available usernames:")
        for u in available:
            print(f"  → https://t.me/{u}")

run_btn = widgets.Button(
    description='🔍 Run Check',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)
run_btn.on_click(run_batch_check)
display(run_btn)

---

## 💡 Tips & Notes

| Tip | Details |
|-----|---------|
| ⏱️ **Delay** | Use ≥ 1.0s to avoid rate limits. Lower = faster but riskier. |
| 🌐 **Proxies** | Essential for bulk checking. Free proxies are unreliable. |
| 📏 **Username Length** | Short names (5-6 chars) are almost all taken. Try 7+. |
| 🔤 **Best Patterns** | Mix letters + numbers: `ab12cd34` > `abcdefgh` |
| 💾 **Results** | Auto-saved to `available_usernames.txt` in the notebook. |
| 📱 **Telegram Alerts** | Set BOT_TOKEN + CHAT_ID to get instant hit notifications. |
| 🤖 **Bot Mode** | Step 3B runs the interactive Telegram bot — great for phone use! |
| 🔄 **Re-run** | You can re-run any cell multiple times with different settings. |

---

### 📌 Telegram Username Rules

```
✅ Valid:   hello, test_123, myname5
❌ Invalid: ab (too short), 1abc (starts with number), abc_ (ends with _), a__b (double __)
```

- **Length:** 5–32 characters
- **Characters:** `a-z`, `0-9`, `_` (underscore)
- **Must start with:** a letter
- **No:** double underscores (`__`)
- **Can't end with:** underscore

---

<div align="center">

### 🙏 Found this useful?

[![GitHub Stars](https://img.shields.io/github/stars/Shineii86/TelegramUserCheckBot?style=social)](https://github.com/Shineii86/TelegramUserCheckBot/stargazers)
[![GitHub Forks](https://img.shields.io/github/forks/Shineii86/TelegramUserCheckBot?style=social)](https://github.com/Shineii86/TelegramUserCheckBot/fork)

**⭐ Star the repo** if it helped you find a great username!

**Made with ❤️ by [@Shineii86](https://github.com/Shineii86)**

</div>